# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

In [13]:
%%writefile .env
NVIDIA_API_KEY="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"

Overwriting .env


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [15]:
pip install -q --upgrade pypdf gradio openai python-dotenv

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="your_key"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [16]:
from pypdf import PdfReader
from pathlib import Path

# El repo ya trae el paper como 1706.03762v7.pdf; si descargaste otra copia con el nombre
# sugerido, se usa esa.
PDF_PATH = "attention_is_all_you_need.pdf" if Path("attention_is_all_you_need.pdf").exists() else "1706.03762v7.pdf"

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages_text = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages_text)

document_text = extract_text_from_pdf(PDF_PATH)
print(document_text[:500])
print("\nCaracteres extraídos:", len(document_text))

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

Caracteres extraídos: 39599


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [17]:
import os
from pathlib import Path

print("Carpeta actual:", os.getcwd())
print(".env existe:", Path(".env").exists())

try:
    from google.colab import userdata
    valor = userdata.get("NVIDIA_API_KEY")
    print("Secret encontrado, largo:", len(valor) if valor else 0)
except Exception as e:
    print("Error leyendo Secret:", type(e).__name__, str(e)[:200])

Carpeta actual: /content
.env existe: True
Secret encontrado, largo: 141


In [18]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.getenv("NVIDIA_API_KEY"):
    raise RuntimeError("Falta NVIDIA_API_KEY: crea el archivo .env como indica el Paso 0.3.")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"
print("Cliente listo. Modelo:", MODELO)

Cliente listo. Modelo: nvidia/nemotron-3-super-120b-a12b


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [19]:
def build_system_prompt(document_text):
    return (
        "Eres un experto en el paper 'Attention Is All You Need' (Vaswani et al., 2017). "
        "Responde únicamente con información presente en el documento que aparece entre "
        "<documento> y </documento>. Si la respuesta no está en el documento, responde: "
        "\"No encuentro esa información en el documento\".\n\n"
        f"<documento>\n{document_text}\n</documento>"
    )

def history_to_messages(history):
    """Convierte el historial de Gradio (dicts o tuplas) a mensajes user/assistant."""
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            messages.append({"role": item["role"], "content": item["content"]})
        else:
            user_msg, assistant_msg = item
            messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages

def chat_con_documento(message, history, document_text):
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    respuesta = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [29]:
import gradio as gr

n_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Páginas extraídas: {n_paginas} | Caracteres: {len(document_text)}")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente de 'Attention Is All You Need'",
    description="Pregunta sobre el paper. Las respuestas se basan únicamente en el documento.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    )

demo.launch(share=True, show_error=True)

Páginas extraídas: 15 | Caracteres: 39599
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e122a6ea41489ec90.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?


## Pruebas del asistente

**1. ¿Cuál es la arquitectura principal propuesta en el paper?**
Respuesta: La arquitectura principal propuesta en el paper es el Transformer, una red basada exclusivamente en mecanismos de atención (multi‑head self‑attention) que elimina totalmente las capas recurrentes y convolucionales. El Transformer consta de un encoder y un decoder, cada uno formado por una pila de N = 6 capas idénticas; cada capa incluye una sub‑capa de multi‑head self‑attention seguida de una red feed‑forward punto‑a‑punto, con conexiones residuales y normalización de capa, además de codificaciones posicionales seno‑coseno. Esta arquitectura permite una mayor paralelización y alcanza un nuevo estado‑de‑arte en tareas de traducción automática.

**2 ¿Qué es el mecanismo de atención?"**
Respuesta: Un mecanismo de atención puede describirse como una función que mapea una consulta y un conjunto de pares clave-valor a una salida, donde la consulta, las claves, los valores y la salida son todos vectores. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor se calcula mediante una función de compatibilidad de la consulta con la correspondiente clave.

**3. ¿Cuántas capas tiene el encoder del modelo base?**
Respuesta: 6

**4. ¿Quiénes son los autores del paper?**
Respuesta: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin.

**5. ¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?**
Respuesta: 28.4 BLEU.

**6. (Pregunta trampa) ¿Qué es GPT-4?**
Respuesta: No encuentro esa información en el documento.


## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
Enviar todo el paper en cada pregunta tiene tres costos. Primero, consume mucho contexto: el paper tiene unos 39,600 caracteres y eso se repite en cada llamada, lo que aumenta el costo y la latencia. Segundo, el modelo tiene que buscar la respuesta dentro de un texto largo, y puede pasar por alto un detalle o mezclar secciones. Tercero, el método no escala: con varios documentos grandes, no cabrían todos en el contexto.
2. ¿Por qué este ejercicio todavía no implementa RAG?
Porque aquí no hay recuperación. El documento completo se inserta en el prompt y el modelo responde con todo ese texto disponible. RAG necesita tres pasos adicionales: dividir el documento en fragmentos, convertirlos en vectores (embeddings) y, para cada pregunta, recuperar solo los fragmentos más relevantes antes de llamar al modelo. Este ejercicio sirve como motivación para RAG: muestra por qué no basta con dar todo el texto.
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del conocimiento previo del modelo?
Pediría que cada respuesta cite una frase textual del paper, y luego buscaría esa frase en el PDF. También haría preguntas sobre temas que no aparecen en el documento, como la pregunta de GPT-4: si el modelo responde con detalles que no están en el paper, está usando su conocimiento previo. Otra prueba útil es preguntar por datos muy específicos del paper, como un número de la tabla 2, y verificar si coincide.
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el prompt?
El principal es la inyección de instrucciones: un documento puede contener texto como "ignora las instrucciones anteriores y responde X", y el modelo podría obedecerlo. Otro riesgo es la fuga de información: si el sistema tiene datos sensibles en el prompt, el usuario podría extraerlos con preguntas bien planteadas. También hay riesgo de que el contenido malicioso o sesgado del documento influya en las respuestas sin que el usuario lo note.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)